# Bronze Layer ETL — Data Ingestion Procedure

## Overview
This Databricks PySpark notebook implements a scalable, production-grade ETL ingestion pipeline for the Bronze layer.

### Process Highlights:
1. **Schema Enforcement**: Ingests raw CSV extracts using explicit PySpark `StructType` definitions to ensure type safety.
2. **Dynamic Path Resolution**: Configures source paths via Databricks widgets (`dbutils.widgets`), environment variables, or local repository fallbacks.
3. **Atomic Overwrite**: Utilizes Delta Lake's `.write.format("delta").mode("overwrite").saveAsTable(...)` to faithfully reproduce the original `TRUNCATE TABLE` + `COPY` batch load behavior.
4. **Observability & Auditing**: Measures load durations per table and total batch, logging row counts and load metrics.

### Source Files:
- `datasets/source_crm/cust_info.csv` -> `bronze.crm_cust_info`
- `datasets/source_crm/prd_info.csv` -> `bronze.crm_prd_info`
- `datasets/source_crm/sales_details.csv` -> `bronze.crm_sales_details`
- `datasets/source_erp/LOC_A101.csv` -> `bronze.erp_loc_a101`
- `datasets/source_erp/CUST_AZ12.csv` -> `bronze.erp_cust_az12`
- `datasets/source_erp/PX_CAT_G1V2.csv` -> `bronze.erp_px_cat_g1v2`

In [ ]:
# Databricks notebook source
# COMMAND ----------
import os
import sys
import time
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DateType, TimestampType
)

# Display helper fallback
try:
    display
except NameError:
    def display(df, n=20):
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

## Dynamic Dataset Path Configuration
Configure the dataset base path using Databricks widgets with automatic fallback to repository relative paths.

In [ ]:
# COMMAND ----------
# Widget configuration with fallback
try:
    dbutils.widgets.text("base_dataset_path", "", "Base Dataset Directory Path")
    widget_val = dbutils.widgets.get("base_dataset_path").strip()
except Exception:
    widget_val = ""

if widget_val:
    BASE_DATASET_PATH = widget_val
elif "DATASET_BASE_PATH" in os.environ:
    BASE_DATASET_PATH = os.environ["DATASET_BASE_PATH"]
else:
    # Search for datasets directory relative to notebook execution
    candidate_dirs = [
        os.path.abspath("datasets"),
        os.path.abspath("../datasets"),
        os.path.abspath("../../datasets"),
        os.path.abspath("../../../datasets"),
        os.path.join(os.getcwd(), "datasets")
    ]
    BASE_DATASET_PATH = "datasets"
    for c_dir in candidate_dirs:
        if os.path.exists(c_dir) and os.path.isdir(c_dir):
            BASE_DATASET_PATH = c_dir
            break

CRM_SOURCE_DIR = os.path.join(BASE_DATASET_PATH, "source_crm")
ERP_SOURCE_DIR = os.path.join(BASE_DATASET_PATH, "source_erp")

print(f"Base Dataset Path : {BASE_DATASET_PATH}")
print(f"CRM Source Path   : {CRM_SOURCE_DIR}")
print(f"ERP Source Path   : {ERP_SOURCE_DIR}")

## PySpark StructType Schemas
Explicit schema definitions matching Bronze table DDL to guarantee reliable CSV parsing.

In [ ]:
# COMMAND ----------
# 1. CRM Customer Info Schema
schema_crm_cust_info = StructType([
    StructField("cst_id", IntegerType(), True),
    StructField("cst_key", StringType(), True),
    StructField("cst_firstname", StringType(), True),
    StructField("cst_lastname", StringType(), True),
    StructField("cst_marital_status", StringType(), True),
    StructField("cst_gndr", StringType(), True),
    StructField("cst_create_date", DateType(), True)
])

# 2. CRM Product Info Schema
schema_crm_prd_info = StructType([
    StructField("prd_id", IntegerType(), True),
    StructField("prd_key", StringType(), True),
    StructField("prd_nm", StringType(), True),
    StructField("prd_cost", IntegerType(), True),
    StructField("prd_line", StringType(), True),
    StructField("prd_start_dt", TimestampType(), True),
    StructField("prd_end_dt", TimestampType(), True)
])

# 3. CRM Sales Details Schema
schema_crm_sales_details = StructType([
    StructField("sls_ord_num", StringType(), True),
    StructField("sls_prd_key", StringType(), True),
    StructField("sls_cust_id", IntegerType(), True),
    StructField("sls_order_dt", IntegerType(), True),
    StructField("sls_ship_dt", IntegerType(), True),
    StructField("sls_due_dt", IntegerType(), True),
    StructField("sls_sales", IntegerType(), True),
    StructField("sls_quantity", IntegerType(), True),
    StructField("sls_price", IntegerType(), True)
])

# 4. ERP Location Schema
schema_erp_loc_a101 = StructType([
    StructField("cid", StringType(), True),
    StructField("cntry", StringType(), True)
])

# 5. ERP Customer AZ12 Schema
schema_erp_cust_az12 = StructType([
    StructField("cid", StringType(), True),
    StructField("bdate", DateType(), True),
    StructField("gen", StringType(), True)
])

# 6. ERP Product Categories Schema
schema_erp_px_cat_g1v2 = StructType([
    StructField("id", StringType(), True),
    StructField("cat", StringType(), True),
    StructField("subcat", StringType(), True),
    StructField("maintenance", StringType(), True)
])

print("All 6 Bronze StructType schemas defined successfully.")

## Bronze Table Ingestion Function
Reusable function to load each CSV file into its corresponding Bronze Delta table with timing and metrics logging.

In [ ]:
# COMMAND ----------
def load_bronze_table(file_path: str, table_name: str, schema: StructType) -> dict:
    """
    Loads a CSV file into a Bronze Delta Lake table.
    
    Args:
        file_path: Absolute or relative path to source CSV.
        table_name: Fully qualified target table name (e.g. 'bronze.crm_cust_info').
        schema: Explicit StructType schema definition.
        
    Returns:
        dict containing table name, record count, duration in seconds, and status.
    """
    start_time = time.time()
    print(f">> Truncating Table: {table_name}")
    print(f">> Inserting Data Into: {table_name} from '{file_path}'")
    
    # Ingest CSV with schema enforcement
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("delimiter", ",")
        .schema(schema)
        .load(file_path)
    )
    
    # Ensure column names are lowercase to match metastore DDL
    for col_name in df.columns:
        if col_name != col_name.lower():
            df = df.withColumnRenamed(col_name, col_name.lower())
            
    # Save to Delta Lake in overwrite mode (equivalent to TRUNCATE + COPY)
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    
    end_time = time.time()
    duration = end_time - start_time
    row_count = df.count()
    
    print(f">> Table {table_name} loaded successfully.")
    print(f">> Load Duration: {duration:.2f} seconds | Row Count: {row_count:,}")
    
    return {
        "table": table_name,
        "rows": row_count,
        "duration_sec": duration,
        "status": "SUCCESS"
    }

## Master Ingestion Procedure: `load_bronze()`
Orchestrates the loading of all 6 Bronze tables, providing automated ingestion and observability across all Bronze Delta tables.

In [ ]:
# COMMAND ----------
def load_bronze() -> list:
    """
    Main pipeline entrypoint for loading the entire Bronze layer.
    Tracks total duration, CRM/ERP sub-phases, and handles errors.
    """
    batch_start_time = time.time()
    load_summary = []
    
    print("================================================")
    print("Loading Bronze Layer")
    print("================================================")
    
    try:
        # Ensure target database exists
        spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
        
        # --- Load CRM Tables ---
        print("\n------------------------------------------------")
        print("Loading CRM Tables")
        print("------------------------------------------------")
        
        # 1. crm_cust_info
        load_summary.append(load_bronze_table(
            os.path.join(CRM_SOURCE_DIR, "cust_info.csv"),
            "bronze.crm_cust_info",
            schema_crm_cust_info
        ))
        
        # 2. crm_prd_info
        load_summary.append(load_bronze_table(
            os.path.join(CRM_SOURCE_DIR, "prd_info.csv"),
            "bronze.crm_prd_info",
            schema_crm_prd_info
        ))
        
        # 3. crm_sales_details
        load_summary.append(load_bronze_table(
            os.path.join(CRM_SOURCE_DIR, "sales_details.csv"),
            "bronze.crm_sales_details",
            schema_crm_sales_details
        ))
        
        # --- Load ERP Tables ---
        print("\n------------------------------------------------")
        print("Loading ERP Tables")
        print("------------------------------------------------")
        
        # 4. erp_loc_a101
        load_summary.append(load_bronze_table(
            os.path.join(ERP_SOURCE_DIR, "LOC_A101.csv"),
            "bronze.erp_loc_a101",
            schema_erp_loc_a101
        ))
        
        # 5. erp_cust_az12
        load_summary.append(load_bronze_table(
            os.path.join(ERP_SOURCE_DIR, "CUST_AZ12.csv"),
            "bronze.erp_cust_az12",
            schema_erp_cust_az12
        ))
        
        # 6. erp_px_cat_g1v2
        load_summary.append(load_bronze_table(
            os.path.join(ERP_SOURCE_DIR, "PX_CAT_G1V2.csv"),
            "bronze.erp_px_cat_g1v2",
            schema_erp_px_cat_g1v2
        ))
        
        batch_end_time = time.time()
        total_duration = batch_end_time - batch_start_time
        
        print("\n==========================================")
        print("Loading Bronze Layer is Completed")
        print(f"   - Total Tables Loaded  : {len(load_summary)}")
        print(f"   - Total Records Ingested: {sum(t['rows'] for t in load_summary):,}")
        print(f"   - Total Load Duration  : {total_duration:.2f} seconds")
        print("==========================================")
        return load_summary
        
    except Exception as err:
        print("\n==========================================")
        print("ERROR OCCURRED DURING LOADING BRONZE LAYER")
        print(f"Error Message: {err}")
        print("==========================================")
        raise err

## Execute Ingestion
Trigger the bronze ingestion pipeline.

In [ ]:
# COMMAND ----------
# Execute the Bronze Layer ingestion procedure
bronze_metrics = load_bronze()

## Bronze Layer Validation & Record Counts
Audit the populated Bronze Delta tables by verifying row counts against expected source counts.

In [ ]:
# COMMAND ----------
# Audit Bronze Delta Tables
expected_counts = {
    "bronze.crm_cust_info": 18494,
    "bronze.crm_prd_info": 397,
    "bronze.crm_sales_details": 60398,
    "bronze.erp_loc_a101": 18484,
    "bronze.erp_cust_az12": 18484,
    "bronze.erp_px_cat_g1v2": 37
}

print("--- Bronze Layer Audit Report ---")
for tbl_name, exp_count in expected_counts.items():
    actual_count = spark.table(tbl_name).count()
    match_status = "MATCH" if actual_count == exp_count else "MISMATCH"
    print(f"[{match_status}] {tbl_name:30}: Actual={actual_count:6,} | Expected={exp_count:6,}")

# Display sample data from each Bronze table
for tbl_name in expected_counts.keys():
    print(f"\nSample data from {tbl_name}:")
    display(spark.table(tbl_name).limit(5))